In [ ]:
import sagemaker
import boto3
from botocore.exceptions import ClientError
import ast
import datasets
import time
import pandas as pd
import os

AWS_ACCESS_KEY_ID=
AWS_SECRET_ACCESS_KEY=
AWS_SESSION_TOKEN=\n
session = sagemaker.Session()

sagemaker_session_bucket=None
if sagemaker_session_bucket is None and session is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = session.default_bucket()
    
role = sagemaker.get_execution_role()

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)

print(f"sagemaker role arn: {role}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
def get_secret_hf(session):

    secret_name = "hf-access-token"
    region_name = "ca-central-1"

    # Create a Secrets Manager client
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )
    
    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return ast.literal_eval(secret)['hf-access-token']

session = boto3.session.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

hf_token = get_secret_hf(session)

In [ ]:
from sagemaker.huggingface import get_huggingface_llm_image_uri

# retrieve the llm image uri
llm_image = get_huggingface_llm_image_uri(
  "huggingface",
  version="1.4.2"
)

# print ecr image uri
print(f"llm image uri: {llm_image}")

In [ ]:
import json
from sagemaker.huggingface import HuggingFaceModel
from sagemaker.async_inference.async_inference_config import AsyncInferenceConfig

# sagemaker config
instance_type = "ml.g5.48xlarge"
number_of_gpu = 1
health_check_timeout = 600

# TGI config
config = {
    'HF_MODEL_ID': "/opt/ml/model",
    # 'HF_MODEL_ID': 'meta-llama/Llama-2-7b-chat-hf',
     # 'HF_MODEL_ID': 'mistralai/Mistral-7B-Instruct-v0.2',
    'SM_NUM_GPUS': json.dumps(number_of_gpu), # Number of GPU used per replica
    'MAX_INPUT_LENGTH': json.dumps(3072),  # Max length of input text
    'MAX_TOTAL_TOKENS': json.dumps(8192),  # Max length of the generation (including input text)
    'MAX_BATCH_TOTAL_TOKENS': json.dumps(8192),  # Limits the number of tokens that can be processed in parallel during the generation
    'HUGGING_FACE_HUB_TOKEN': hf_token,
    'HF_API_TOKEN': hf_token,
    
}

async_config = AsyncInferenceConfig(
    output_path= "s3://eko-ekoka-ai-project/data/generation_output/Apr12_generation_1" ,
)


# create HuggingFaceModel
llm_model = HuggingFaceModel(
    role=role,
    image_uri=llm_image,
    # model_data="s3://eko-ekoka-ai-project/output/ka-finetuning-2024-04-12-19-19-50-218/output/model.tar.gz", # Mistral 7B Fine-tuned Only
    env=config,
    

)

async_predictor = llm_model.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    async_inference_config=async_config,
    tags=[{"Key":'uw-ai-inference', "Value":'initial-model-inference'}],
    vpc_config_override = {"Subnets": ['subnet-0ba981d4c6314f9a9'], "SecurityGroupIds": ['sg-04645476ef00795e3'] },
    
)


In [ ]:
# llm = llm_model.deploy(
#     initial_instance_count=1,
#     instance_type=instance_type,
#     container_startup_health_check_timeout=health_check_timeout, # 10 minutes to be able to load the model
#     tags=[{"Key":'uw-ai-inference', "Value":'initial-model-inference'}],
# )


In [ ]:
def build_llama_prompt(content, system_prompt = 'default' ):
    start_prompt = "<s>[INST] "
    end_prompt = " [/INST]"
    if system_prompt == 'default':
        system_prompt = f"""<<SYS>>\nYou are a helpful, respectful, and honest assistant in a pediatric rehabilitation clinic. You follow these rules:
1. Follow directions meticulously.
2. Write in point-form. Do not write in paragraphs.
3. Do not produce any extra text. Only write what is asked for.
4. Clearly distinguish between reporting, observations, and goals.
5. Answer as helpfully as possible, while being safe. <</SYS>>
"""

    
    return start_prompt + system_prompt + content + end_prompt

In [ ]:
def build_mistral_prompt(content, system_prompt = 'default' ):
    start_prompt = "<s>[INST] "
    end_prompt = " [/INST]"
    if system_prompt == 'default':
        system_prompt = f"""You are a helpful, respectful, and honest assistant in a pediatric rehabilitation clinic. You follow these rules:
1. Follow directions meticulously.
2. Write in point-form. Do not write in paragraphs.
3. Do not produce any extra text. Only write what is asked for.
4. Clearly distinguish between reporting, observations, and goals.
5. Answer as helpfully as possible, while being safe.
"""

    
    return start_prompt + system_prompt + content + end_prompt

In [ ]:
def generate_response(full_prompt, top_p = 0.9, temp = 0.8, max_tokens = 1024, rep_penalty = 1.03, stop_list=["<|endoftext|>","</s>", "OT Reg.(Ont.)", "OT Reg. (Ont.)"]):
    
    # input_ids = llama_tokenizer(full_prompt)["input_ids"]
    # num_input_tokens = len(input_ids)
    # max_tokens = 4096 - num_input_tokens
    
    payload = {
      "inputs":  full_prompt,
      "parameters": {
        "do_sample": True,
        "top_p": top_p,
        "temperature": temp,
        "max_new_tokens": max_tokens,
        "repetition_penalty": rep_penalty,
        "stop": stop_list,
        
      }
    }

    # send request to endpoint
    response = async_predictor.predict(payload)

    print(response[0]["generated_text"][len(full_prompt):])
    return response

In [ ]:
def extract_generated_output(text):
    text = str(text)
    split_text = text.split('[/INST]', 1)
    if len(split_text) > 1:
        return split_text[1].strip()
    else:
        return ""
    
def create_generation_output_df(df, i=-1):
    if i == -1:
        df['generated_output'] = df['output'].apply(extract_generated_output)
        df['generated_output'] = df['generated_output'].str.replace(r'"}]', '')
    else:
        df[f'generated_output_{i}'] = df[f'output_{i}'].apply(extract_generated_output)
        df[f'generated_output_{i}'] = df[f'generated_output_{i}'].str.replace(r'"}]', '')
    return df

In [ ]:
curated = pd.read_csv('data/curated_examples.csv', encoding='MacRoman')
curated = curated.dropna(axis=1, how='all')

prefix_PN = "This is a Progress Note - OT from "
prefix_SN = "This is a Scratch Note - OT from "
# curated['PN'] = curated.apply(lambda row: prefix_PN + row['Date'] + ":\n" + row['PN'], axis=1)
curated['SN_Input'] = curated.apply(lambda row: prefix_SN + row['Date'] + ":\n" + row['SN'], axis=1)

curated_sample = curated.sample(n=10, random_state=50)


In [ ]:
def evaluate_on_sample(filename, df, prompt_content, repeat = 1, max_new_tokens=1024):
    
    sn_list = df['SN_Input']
    df = df.drop('SN_Input', axis=1)
    
    for j in range(0, repeat):
        responses = []
        time_ids = []

        for sn in sn_list:
            # prompt = build_llama_prompt(f'{prompt_content} \n{sn}', 'default')
            prompt = build_mistral_prompt(f'{prompt_content} \n{sn}', 'default')
            response = generate_response(prompt, max_tokens=max_new_tokens)
            responses.append(response)
            timestr = time.strftime("%Y_%m_%d-%H%M%S_")
            time_ids.append(timestr)

        df[f'output_{j}'] = responses
        df[f'time_{j}'] = time_ids
        
        df_new = create_generation_output_df(df, j)
   
    if os.path.exists(filename):
        # If the file exists, append an integer to the filename until it's unique
        i = 1
        while True:
            new_filename = f"{os.path.splitext(filename)[0]}_{i}.csv"
            if not os.path.exists(new_filename):
                filename = new_filename
                break
            i += 1


    df_new.to_csv(filename, index=False)
    return df_new
        


In [ ]:
def evaluate_multiple_generations(filename, df, column_prefix, max_new_tokens=500):
    
    responses = []
    time_ids = []
    
    for _, row in df.iterrows():
        values = [row[col] for col in row.index if col.startswith(column_prefix)]

        sn = row['SN']
        
        content = f"""Rank the Generated Progress Notes based on the following criteria: 1. How readable the Progress Note is. 2. How appropriate the format is. It should be a SOAP note with clear language that does not repeat itself. 3. How faithfully the Progress Note captures the content in the original Scratch Note.
All the Generated Progress Notes are based on the following original Scratch Note: {sn}
"""
        i=1
        for pn in values:
            i+=1
            content += f"""Generated Scratch Note {i}: {pn}\n"""
        
        content += f"""Output only the best Generated Progress Note."""
        
    
        # prompt = build_llama_prompt(f'{content} \n{sn}', 'default')
        prompt = build_mistral_prompt(f'{content} \n{sn}', 'default')
        response = generate_response(prompt, max_tokens=max_new_tokens)
        responses.append(response)
        timestr = time.strftime("%Y_%m_%d-%H%M%S_")
        time_ids.append(timestr)
        
    
    df['output'] = responses
    df[f'ranking_time'] = time_ids
    
    df_new = create_generation_output_df(df)
    
    
    if os.path.exists(filename):
        # If the file exists, append an integer to the filename until it's unique
        i = 1
        while True:
            new_filename = f"{os.path.splitext(filename)[0]}_{i}.csv"
            if not os.path.exists(new_filename):
                filename = new_filename
                break
            i += 1
            
            
    df_new.to_csv(filename, index=False)
    return df_new
        
    


In [ ]:
csv = 'data/Apr12_generation.csv'
instruction = f'Write a Progress Note in SOAP format based on the following Scratch Note:'

response_df = evaluate_on_sample(csv, curated_sample, instruction, repeat = 3, max_new_tokens = 800)

In [ ]:
csv = 'data/Apr12_rank_generation.csv'

rank_response_df = evaluate_multiple_generations(csv, response_df, 'generated_output', max_new_tokens=800)

In [ ]:
generated_responses = response_df['generated_output']

In [ ]:
response_df = pd.read_csv('data/Mar30_rank_generation.csv')

In [ ]:
format_responses = []
time_ids = []
for gen_response in generated_responses:
    format_instruction = f"Re-format the following progress note to be in bullet points with newline spaces between. Do not change the content. Only change the formatting to be more readable. The Progress Note is \n {gen_response}"
    prompt = build_llama_prompt(format_instruction)
    response = generate_response(prompt, max_tokens=1024)
    format_responses.append(response)
    timestr = time.strftime("%Y_%m_%d-%H%M%S_")
    time_ids.append(timestr)

In [ ]:
response_df['output_rf'] = format_responses
# response_df[f'time'] = time_ids
    
response_df = create_generation_output_df(response_df, i='rf')

In [ ]:
response_df.to_csv('data/Apr1_generation_formatting.csv')

In [ ]:
time_ids

# Doing it manually

In [ ]:
import pandas as pd
curated = pd.read_csv('data/curated_examples.csv', encoding='MacRoman')
curated = curated.dropna(axis=1, how='all')

prefix_PN = "This is a Progress Note - OT from "
prefix_SN = "This is a Scratch Note - OT from "
# curated['PN'] = curated.apply(lambda row: prefix_PN + row['Date'] + ":\n" + row['PN'], axis=1)
curated['SN_Input'] = curated.apply(lambda row: prefix_SN + row['Date'] + ":\n" + row['SN'], axis=1)

In [ ]:
curated_sample = curated.sample(n=10, random_state=50)

In [ ]:
curated_sample

In [ ]:
# ! pip install sentencepiece

In [ ]:
sn_list = curated_sample['SN_Input']
responses = []
time_ids = []

i = 0
for sn in sn_list:
    prompt = build_llama_prompt(f'Write a Progress Note in SOAP format based on the following Scratch Note. Use Bullet Points. \n {sn}', 'default')
    response = generate_response(prompt)
    responses.append(response)
    print(f'Response {i} completed')
    timestr = time.strftime("%Y_%m_%d-%H%M%S_")
    time_ids.append(timestr)
    
    i += 1
    
    
    
    
    

In [ ]:
curated_sample['output'] = responses
curated_sample['time'] = time_ids
curated_sample = curated_sample.drop('SN_Input', axis=1)

In [ ]:
curated_sample['output'][1]

In [ ]:
df_output = create_generation_output_df(curated_sample)

In [ ]:
df_output['generated_output'][1]

In [ ]:
df_output.to_csv('data/Mar25_generation_4.csv')

